In [7]:
import pandas as pd
import numpy as np
from sklearn.ensemble import  GradientBoostingRegressor # Changed from RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error # Removed accuracy_score as it's for classification


train = pd.read_csv("https://raw.githubusercontent.com/tiwari91/Housing-Prices/master/train.csv")
test  = pd.read_csv("https://raw.githubusercontent.com/tiwari91/Housing-Prices/master/test.csv")


print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(train.isnull().sum().sort_values(ascending = False).head(10))
print(test.isnull().sum().sort_values(ascending = False).head(10))

print((train.isnull().sum().sort_values(ascending = False).head(10) / len(train) * 100).round(1))
print((test.isnull().sum().sort_values(ascending = False).head(10) / len(test) * 100).round(1))

test_ids= test['Id']
test['SalePrice'] =  -1

# Remove outliers — large houses with suspiciously low prices
train = train[~((train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000))]
combine = pd.concat([train,test], axis = 0).reset_index(drop = True)

# Total square footage — one of the strongest predictors
combine['TotalSF'] = combine['TotalBsmtSF'] + combine['1stFlrSF'] + combine['2ndFlrSF']

# Total bathrooms
combine['TotalBath'] = (combine['FullBath'] + combine['HalfBath'] * 0.5 +
                        combine['BsmtFullBath'] + combine['BsmtHalfBath'] * 0.5)

# House age
combine['HouseAge'] = combine['YrSold'] - combine['YearBuilt']

# Was it remodeled?
combine['IsRemodeled'] = (combine['YearRemodAdd'] != combine['YearBuilt']).astype(int)

cols_to_drop = ['PoolQC','MiscFeature','Alley','Fence','MasVnrType','FireplaceQu']
combine = combine.drop(columns = cols_to_drop)

numeric_cols = combine.select_dtypes(include = 'number').columns
for col in numeric_cols:
  combine[col] = combine[col].fillna(combine[col].median())

text_cols = combine.select_dtypes(include = 'object').columns
for col in text_cols:
  combine[col] = combine[col].fillna(combine[col].mode()[0])


# The original code did not create 'combined' in the way it's used later,
# which might lead to dimension mismatch errors if not handled.
# For a proper fix, `pd.get_dummies` should be applied after splitting train/test again
# or applied to the combine dataframe and then split.
# Assuming the intent was to apply `get_dummies` to `combine` before splitting.
combined = pd.get_dummies(combine, drop_first = True)

X_train_full, X_test_final = X_train_full.align(X_test_final,
                                                  join='left',
                                                  axis=1,
                                                  fill_value=0)

# Re-extracting train and test data after one-hot encoding for consistent features
X_train_full = combined[combined['SalePrice'] != -1]
y_train_full = X_train_full['SalePrice'] # Ensure y_train_full has correct shape
X_train_full = X_train_full.drop(columns = ['SalePrice'])
X_test_final = combined[combined['SalePrice'] == -1].drop(columns = ['SalePrice'])


y_train_log = np.log(y_train_full)

# Split the training data into training and validation sets
X_train, X_test, y_train, y_test = train_test_split(X_train_full, y_train_log, test_size=0.2, random_state=42)

# Initialize and train a RandomForestRegressor model
model = GradientBoostingRegressor(n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=42)
model.fit(X_train, y_train)

# Make predictions on the test set
predictions = model.predict(X_test)

# Calculate RMSE using the actual y_test and the predictions
rmse = np.sqrt(mean_squared_error(y_test, predictions)) # Removed np.log as y_train_log is already log-transformed
print(f"RMSE: {rmse}")

# Predict on real test set
test_predictions = np.exp(model.predict(X_test_final))

# Save submission
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': test_predictions
})
submission.to_csv('submission.csv', index=False)
print(submission.head(10))
print("Submission shape:", submission.shape)

Train shape: (1460, 81)
Test shape: (1459, 80)
PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageQual        81
GarageFinish      81
GarageType        81
dtype: int64
PoolQC          1456
MiscFeature     1408
Alley           1352
Fence           1169
MasVnrType       894
FireplaceQu      730
LotFrontage      227
GarageYrBlt       78
GarageCond        78
GarageFinish      78
dtype: int64
PoolQC          99.5
MiscFeature     96.3
Alley           93.8
Fence           80.8
MasVnrType      59.7
FireplaceQu     47.3
LotFrontage     17.7
GarageQual       5.5
GarageFinish     5.5
GarageType       5.5
dtype: float64
PoolQC          99.8
MiscFeature     96.5
Alley           92.7
Fence           80.1
MasVnrType      61.3
FireplaceQu     50.0
LotFrontage     15.6
GarageYrBlt      5.3
GarageCond       5.3
GarageFinish     5.3
dtype: float64
RMSE: 0.127984570210429
     Id      SalePrice
0  1461  1

In [8]:
from google.colab import files
files.download('submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>